> Steps Required:
1. Get the original sequence 
2. Translate original sequence
3. Trim spacers from original sequence
4. utilize the AnarcII on the sequence
5. Import the CDR3 from the anarchii alligned sequence to the original sequence
6. Profit

In [9]:
from scripts.helpers import codon_dict

def fast_translate(nt_seq:str) -> str:
    length = len(nt_seq)
    seq_2t = nt_seq[:length - (length % 3)]
    aa_seq = []

    for i in range(1, int(len(seq_2t)/3 + 1)):
        try:
            aa_seq.append(codon_dict[seq_2t[i*3-3:i*3]])
        except:
            aa_seq.append("-")

    return "".join(aa_seq)

In [10]:
from scripts.helpers import translateNT
import pandas as pd 
import os 

n = 100

input_path = os.path.join("input", "cleaned_seqs_all_seq_1k.csv")
input_df = pd.read_csv(input_path, index_col=0)
input_totest = input_df.head(n).germline.str.replace("-","")

In [ ]:

from scripts.helpers import translateNT, read_json
from scripts.anarcii import aligned_frame
from anarcii import Anarcii
import pandas as pd 
import numpy as np
import regex as re
import os  



################################
# IMGT numbring positions scheme
# source -> https://www.imgt.org/IMGTScientificChart/Nomenclature/IMGT-FRCDRdefinition.html
imgt_dict = {range(1,27):"fr1",
             range(27,39):"cdr1",
             range(39,56):"fr2",
             range(56,66):"cdr2",
             range(66,105):"fr3",
             range(105,118):"cdr3",
             range(118,129):"fr4"}


######################################################
# Custom function for region extraction per int number
def get_region(position_aa: int, 
               regions_dict: dict = imgt_dict) -> str:
    """
    Function that returns IMGT heavy chain position according to amino acid position input in int format.
    position_aa: int -> amino acid position.
    regions_dict: dictionary -> IMGT numbring scheme in range dict format.
    """
    for key_range, value in regions_dict.items():
        if position_aa in key_range:
            return value
    return np.nan


#####################################
def get_digit(pos_string:str) -> int:
    """
    Custom function that extract amino acid position in int format from string (for example, extract `111` from `111A`)
    pos_string:str -> amino acid position in string format"
    """
    re_found = re.search(pattern = r"[\d]+", 
                         string=pos_string)

    return int(re_found.group(0))


###################
class AssignCDR3():
    def __init__(self, 
                 input_df:pd.DataFrame | str, 
                 germline_column:str,
                 seq_column:str,
                 results_path:str,
                 append_cdr3: tuple = (True, "cdr3_aa"),
                 model_seqtype:str = "antibody", 
                 model_mode: str = "accuracy"):
        
        """
        Custom class that takes a NT sequence column from dataframe and assign an AnarcII allighen CDR3 to it.

        input_df: pd.DataFrame | str -> Dataset in pd.DataFrame format or exact string path to the input dataset in CSV format.
        germline_column: str -> string name of the column which contains the germline NT sequence.
        seq_column: str -> string name of the column which contains the sequencing NT sequence.
        append_cdr3: tuple with boolean value and string, if True will try to append the cdr3 aa sequence originated from the specified column into the translated sequence.
        model_seqtype:str -> Which model the anarcii algorithm wil load.
        model_mode:str -> Processing mode of the anarcii algorithm.
        """

        # Defining column names
        self.germline_column = germline_column
        self.seq_column = seq_column
        self.append_cdr3 = append_cdr3
        self.results_path = results_path

        # Getting CDR3 information if required
        if append_cdr3[0]:
            self.cdr3_aa_col = append_cdr3[1] 


        # Importing the raw data into python
        if isinstance(input_df, pd.DataFrame):
            self.input_df = input_df

        elif isinstance(input_df, str):
            try:
                self.input_df = pd.read_csv(input_df, index_col=0)

            except:
                raise Exception(f"> Invalid input path (under `input_df` argument): `{input_df}`")

        self.anarchii_model = Anarcii(seq_type=model_seqtype, mode=model_mode)


    def anarchii_allign(self):
        """
        Initializing the class, getting the raw data, and alligining the sequence according to  
        the AnacrII algorithm.
        """
        # Getting the germline NT column -> translating to AA
        germline_nt = self.input_df[self.germline_column]
        germline_aa = germline_nt.apply(translateNT,aa_end=104).str.replace("-","").str.replace("X","")

        if self.append_cdr3[0]:
            germline_aa = germline_aa + self.input_df[self.append_cdr3[1]]

        # Creating anarcii results dataframe
        sequence_anarcii = self.anarchii_model.number(germline_aa)
        self.results_df = pd.DataFrame(sequence_anarcii).T


        # inserting alligned sequence to the results dataframe
        results_anarcii = aligned_frame(sequence_anarcii)
        results_anarcii.columns = pd.MultiIndex.from_tuples([(i, get_region(get_digit(i))) for i in results_anarcii.columns],
                                                            names=["position", "region"])

        results_anarcii.to_csv(os.path.join(self.results_path, "anarcii_sequences.csv"))

        self.results_df.insert(loc=1, 
                               column="germline_anarcii", 
                               value= results_anarcii.astype(str).agg("".join, axis=1))

        self.results_df.insert(loc=2,
                               column="cdr3_anacrii",
                               value=results_anarcii[[i for i in results_anarcii.columns if i[1] == "cdr3"]].astype(str).agg("".join, axis=1))
        
        
        self.results_df.to_csv(os.path.join(self.results_path, "anarcii_output.csv"))

        return self.results_df

        
    def implant_cdr3(self):
        """
        Inserting the AnarcII-alligned CDR3 sequence into the ImmuneDB seq (from aa position 104 onward).
        """
        # seq_id, sample_id, subject_id, clone_id, functional, germline_nt, sequance_nt, germline_aa, cdr3_aa, cdr3_anarcii
        
        final_output = pd.concat([self.input_df[["seq_id", "sample_id", "subject_id", "clone_id", "functional", "copy_number","germline", "sequence", "cdr3_aa"]],
                                      self.results_df[["germline_anarcii", "cdr3_anacrii"]]],
                                      axis=1)
        final_output = final_output[["seq_id", "sample_id", "subject_id", "clone_id", "functional", "copy_number","sequence","germline", "germline_anarcii", "cdr3_aa", "cdr3_anacrii"]]

        final_output.to_csv(os.path.join(self.results_path, "sequences_results.csv"))
        return final_output
                                      

What column do i need in the output? 
> from input_df:
1. seq_id
2. sample_id
3. subject_id
4. clone_id
5. sequence -> sequence_nt
6. germline -> germline_nt
7. cdr3_aa -> cdr3_unalligned

> from anarci results:
1. full translated & alligned sequence -> sequence_aa_anarcii
2. alligned cdr3 aa sequence -> cdr3_alligned

> Type of outputs to save:
1. AnarcII results dataframe 
2. AnacarII alligned sequences dataframe
3. Unified results dataframe

> What to do?



In [12]:
spacer = "-------------------------------------------------------------------------"

# loading run information from `config.json`
print(spacer,"\n> Loading configuration from `config.json`")
config_settings = read_json("config.json")["settings"]
config_cdr3 = read_json("config.json")["AssignCDR3"]
input_dir, input_file, run_name = config_settings["dir_input"], config_settings["file_input"], config_settings["run_name"]
germline_col, sequence_col, cdr3_col = config_cdr3["germline_column"], config_cdr3["sequence_column"], config_cdr3["cdr3_column"]

# Defining paths and creating folders
input_file = os.path.join(input_dir, input_file)
output_dir = os.path.join("results", run_name)
os.makedirs(output_dir, exist_ok=True)

# Initiating the AnarcII class
print(spacer,"\n> Initiating AnarcII class")
aclass = anarcii_class = AssignCDR3(pd.read_csv(input_file, index_col=0),#.head(10), 
                           germline_column=germline_col, 
                           seq_column=sequence_col,
                           results_path=output_dir)

# Performing the alligment 
print(spacer,"\n> Performing alligment")
anarci_allign = aclass.anarchii_allign()

# Unifing data to our original table
print(spacer,"\n> Unifing Data")
anarci_cdr3 = aclass.implant_cdr3()

# Printing final messege + showing saved files
output_path = os.path.join("results", run_name)
print(spacer,f"\n> Processing done. Output files saved at `{output_path}`.")
print(f"> output files: {os.listdir(output_path)}")

------------------------------------------------------------------------- 
> Loading configuration from `config.json`
------------------------------------------------------------------------- 
> Initiating AnarcII class
------------------------------------------------------------------------- 
> Performing alligment
------------------------------------------------------------------------- 
> Unifing Data
------------------------------------------------------------------------- 
> Processing done. Output files saved at `results\test_anarcii`.
> output files: ['anarcii_output.csv', 'anarcii_sequences.csv', 'sequences_results.csv']


In [14]:
import os
import pandas as pd

path_file = os.path.join("input", "covid_vaccine_new.sequences.txt")
df_test = pd.read_csv(path_file, sep="\t")
df_test

,sample_id,ai,subject_id,seq_id,partial,rev_comp,probable_indel_or_misalign,locally_aligned,deletions,insertions,...,stop,copy_number,cdr3_num_nts,cdr3_nt,cdr3_aa,sequence,quality,germline,clone_id,mutations_from_clone
0,1,1,1,M03592:154:000000000-JCY9V:1:1102:6414:23287,1,1,0,1,NaN,NaN,...,0,1,42,TGTGCGAGACTGTTTCAGTTCTACTACNACATGGACGTCTGG,CARLFQFYYXMDVW,NNNNNNNNNNNNNNNNNNNNNNNNNNN---NNNNNNNNNNNNNNNN...,NaN,GAGGTGCAGCTGGTGCAGTCTGGAGCA---GAGGTGAAAAAGCCCG...,NaN,"{""163"": ""nonconservative"", ""201"": ""conservativ..."
1,1,2,1,M03592:154:000000000-JCY9V:1:1110:10855:5783,1,1,0,1,NaN,NaN,...,1,1,68,TGTGCGAGAGGCATGGCTATGGTTGGATATAGTTCTGGGTGAGATA...,CARGMAMVGYSSG*DTWAVLTT,NNNNNNNNNNNNNNNNNNNNNNNNNNN---NNNNNNNNNNNNNNNN...,NaN,CAGGTGCAGCTACAGCAGTGGGGCGCA---GGACTGTTGAAGCCTT...,NaN,"{""112"": ""conservative"", ""122"": ""nonconservativ..."
2,1,3,1,M03592:154:000000000-JCY9V:1:1113:7945:17437,1,1,0,1,229-18,NaN,...,0,1,54,TGTGCGAGAGGTTTTCGGNAACATTATTATGGTTCGGGGAGTTATG...,CARGFRXHYYGSGSYXYW,NNNNNNNNNNNNNNNNNNNNNNNNNNN---NNNNNNNNNNNNNNNN...,NaN,CAGGTGCAGCTGGTGGAGTCTGGGGGA---GGCGTGGTCCAGCCTG...,NaN,"{""81"": ""conservative"", ""106"": ""conservative"", ..."
3,1,4,1,M03592:154:000000000-JCY9V:1:1118:6560:10505,1,1,0,1,NaN,NaN,...,0,1,66,TGTGCCNNAGCCCCTCTTCACGAGACCAGGGGCTGGNNGGGACGGA...,CAXAPLHETRGWXGRIRFFDYW,NNNNNNNNNNNNNNNNNNNNNNNNNNN---NNNNNNNNNNNNNNNN...,NaN,CAGCTGCAGCTGCAGGAGTCCGGCTCA---GGACTGGTGAAGCCTT...,NaN,"{""82"": ""nonconservative"", ""100"": ""conservative..."
4,1,5,1,M03592:154:000000000-JCY9V:1:2103:23409:14161,1,1,0,1,NaN,NaN,...,0,1,75,TGTGCGAGCCCGGGGCCCCTATNTTGTAATAGTATCAGCTGCCATT...,CASPGPLXCNSISCHSGYYNGMDVW,NNNNNNNNNNNNNNNNNNNNNNNNNNN---NNNNNNNNNNNNNNNN...,NaN,CAGGTGCAGCTGGTGGAGTCTGGGGGA---GGCTTGGTCAAGCCTG...,NaN,"{""86"": ""synonymous"", ""103"": ""conservative"", ""1..."
5,1,6,1,M03592:154:000000000-JCY9V:1:2105:15012:16999,1,1,0,1,160-13,NaN,...,0,1,62,TGTACAAAAAGGGGCCTCCGAAATTGTAGTGGTGGTTGCTGCACAA...,CTKRGLRNCSGGCCTTGPXP,NNNNNNNNNNNNNNNNNNNNNNNNNNN---NNNNNNNNNNNNNNNN...,NaN,GAGGTGCAGCTGGTGGAGTCTGGGGGA---GGCTTGGTACAGCCTG...,NaN,"{""107"": ""synonymous"", ""108"": ""nonconservative""..."
6,1,7,1,M03592:154:000000000-JCY9V:1:2117:20197:5409,1,1,0,1,NaN,NaN,...,0,1,45,TGTGCAACTCGAATTCCATGGAGATATGATGCTTTTGATATCTGG,CATRIPWRYDAFDIW,NNNNNNNNNNNNNNNNNNNNNNNNNNN---NNNNNNNNNNNNNNNN...,NaN,CAGGTGCAGCTGGTGCAGTCTGGGGCT---GAGGTGAAGAAGCCTG...,NaN,"{""237"": ""nonconservative"", ""238"": ""conservativ..."
7,2,8,2,M03592:154:000000000-JCY9V:1:1103:25811:22501,1,1,0,1,NaN,NaN,...,0,1,42,NGTGCGAGACCGGGTGGGGTGGNTGGCCCTTTTGACTACTGG,XARPGGVXGPFDYW,NNNNNNNNNNNNNNNNNNNNNNNNNNN---NNNNNNNNNNNNNNNN...,NaN,CAGGTCCAGCTTGTGCAGTCTGGGGCT---GAGGTGAAGAAGCCTG...,NaN,"{""103"": ""conservative"", ""106"": ""nonconservativ..."
8,2,9,2,M03592:154:000000000-JCY9V:1:1105:18940:12984,1,1,0,1,NaN,NaN,...,0,1,72,TGTGCGAGACAGTCCGGGGGATTTTGCTCTGGTGGTAGGTGCTTCG...,CARQSGGFCSGGRCFAGRXWLDPW,NNNNNNNNNNNNNNNNNNNNNNNNNNN---NNNNNNNNNNNNNNNN...,NaN,CAGGTGCAGCTGCAGGAGTCGGGCCCA---GGACTGGTGAAGCCTT...,NaN,"{""103"": ""nonconservative"", ""123"": ""conservativ..."
9,2,10,2,M03592:154:000000000-JCY9V:1:2104:6381:4403,1,1,0,1,NaN,NaN,...,0,1,42,TGTGCGAGGTGTTTTGGTGGTGGCTGCTACCTTGACTACTGG,CARCFGGGCYLDYW,NNNNNNNNNNNNNNNNNNNNNNNNNNN---NNNNNNNNNNNNNNNN...,NaN,CAGGTGCAGCTGGTGGAGTCTGGGGGA---GGCGTGGTCCAGCCTG...,NaN,"{""262"": ""nonconservative"", ""287"": ""synonymous""..."
